# MDR-CLS v1.0.1
**Temporal-Station Baseline Model for Soil Moisture Prediction**

**Author:**  
**Affiliation:** Seattle University, Computer Science  
**Project:** MDR
**Notebook Type:** Training & Evaluation  
**Last Updated:** May 18th 2026

---

## Model Summary
- **Model Name:** MDR-CLS 
- **Version:** v1.0.1
- **Task:** Classification (Regime)

---

## Reproducibility
- **Random Seed:** 42
- **Split Metadata:** `data/splits/derived_9.0/split_meta.json`
- **Environment:** Jupyter Lab + VS Code

---

**What's new?**

- Classifier for regime change detection added (v20.4)
- Use derived_9.0 splits with more stations (v20.4.2)


**Simpson's pradox**

## 0. Imports

In [1]:
import os
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, balanced_accuracy_score
)

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

project_root = os.path.abspath("../../")
if project_root not in sys.path:
    sys.path.append(project_root)

from Models.Utils.dashboard import metrics_dashboard

import warnings
warnings.filterwarnings("ignore")

print("imports loaded")

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

imports loaded


In [2]:
SEED = 42
DEEP_SEARCH = 40

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"Random seed set to {SEED}")

def print_env_info():
    print("Environment information:")
    print(f"  Python version: {os.sys.version.split()[0]}")
    print(f"  NumPy version:  {np.__version__}")
    print(f"  Pandas version: {pd.__version__}")

    try:
        import xgboost
        print(f"  XGBoost version: {xgboost.__version__}")
    except ImportError:
        print("  XGBoost not installed")

    IN_COLAB = "COLAB_GPU" in os.environ
    print(f"  Running in Colab: {IN_COLAB}")

    if IN_COLAB:
        gpu = os.environ.get("COLAB_GPU", None)
        print(f"  GPU available: {gpu}")
    else:
        print("  GPU available: False")

print_env_info()

plt.style.use("default")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

print("environment setup complete")

Random seed set to 42
Environment information:
  Python version: 3.12.8
  NumPy version:  2.4.4
  Pandas version: 3.0.3
  XGBoost version: 3.2.0
  Running in Colab: False
  GPU available: False
environment setup complete


In [3]:
VERSION = "v20"
SUBVERSION = "v20.4.2"
RUN_NAME = "mdr_ts_v20_4.2"

PROJECT_ROOT = os.path.abspath("../../")
DATA_ROOT = f"{PROJECT_ROOT}/Temporal/Pipeline/data"
SPLIT_ROOT = f"{DATA_ROOT}/splits"
OUTPUT_ROOT = f"{PROJECT_ROOT}/Models/Temporal/{VERSION}/{SUBVERSION}"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Project paths:")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  DATA_ROOT:    {DATA_ROOT}")
print(f"  SPLIT_ROOT:   {SPLIT_ROOT}")
print(f"  OUTPUT_ROOT:  {OUTPUT_ROOT}")

print("\nKey file checks:")
print("  data exists:",
      os.path.exists(DATA_ROOT))
print("  splits exists:",
      os.path.exists(SPLIT_ROOT))
print("  output exists:",
      os.path.exists(OUTPUT_ROOT))

Project paths:
  PROJECT_ROOT: C:\Users\pan\Documents\GitHub\MDR-Project
  DATA_ROOT:    C:\Users\pan\Documents\GitHub\MDR-Project/Temporal/Pipeline/data
  SPLIT_ROOT:   C:\Users\pan\Documents\GitHub\MDR-Project/Temporal/Pipeline/data/splits
  OUTPUT_ROOT:  C:\Users\pan\Documents\GitHub\MDR-Project/Models/Temporal/v20/v20.4.2

Key file checks:
  data exists: True
  splits exists: True
  output exists: True


In [4]:
TRAIN_PATH = str(Path(SPLIT_ROOT) / "derived_9.0/train.csv")
VAL_PATH   = str(Path(SPLIT_ROOT) / "derived_9.0/val.csv")
TEST_PATH  = str(Path(SPLIT_ROOT) / "derived_9.0/test.csv")

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing split file: {p}")

print("Split files:")
print(" ", TRAIN_PATH)
print(" ", VAL_PATH)
print(" ", TEST_PATH)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

Split files:
  C:\Users\pan\Documents\GitHub\MDR-Project\Temporal\Pipeline\data\splits\derived_9.0\train.csv
  C:\Users\pan\Documents\GitHub\MDR-Project\Temporal\Pipeline\data\splits\derived_9.0\val.csv
  C:\Users\pan\Documents\GitHub\MDR-Project\Temporal\Pipeline\data\splits\derived_9.0\test.csv


In [5]:
print("Common columns across splits:",
      len(set(train_df.columns) & set(val_df.columns) & set(test_df.columns)))

print("\ncolumns:")
print(list(train_df.columns)[:30])

Common columns across splits: 499

columns:
['station_id', 'date', 'longitude', 'latitude', 'precip_mm', 's1_vv', 's1_vh', 's2_b4', 's2_b8', 's2_b11', 's2_b12', 'LST_modis', 'elev', 'slope', 'aspect', 'DOY', 'SMAP_sm_am_interp', 'SMAP_sm_pm_interp', 'soil_moisture_5cm', 'J_aspect_deg', 'J_bio_bio01', 'J_bio_bio02', 'J_bio_bio03', 'J_bio_bio04', 'J_bio_bio05', 'J_bio_bio06', 'J_bio_bio07', 'J_bio_bio08', 'J_bio_bio09', 'J_bio_bio10']


In [6]:
TARGET_COL = "soil_moisture_5cm"
KEEP_META_COLS = ["station_id", "date", "longitude", "latitude"]

FEATURE_COLS_BASE = [
    "SMAP_sm_pm_interp_ema02",
    "SMAP_sm_interp_grad7",
    "SMAP_ampm_diff_interp",

    "SMAP_sm_interp_lag1",
    "SMAP_sm_interp_diff1",
    "SMAP_sm_interp_rollmean7",

    "G_API",
    "G_rain_sum_3d",
    "G_rain_sum_7d",
    "V_ema_G_API_kobs7",
    "V_ema_G_API_kobs14",
    "V_ema_G_API_kobs30",
    "V_rollmean_G_API_kobs7",
    "V_rollmean_G_API_kobs14",

    "V_rollcv_G_API_kobs7",
    "A_d_G_API_kobs7",

    "A_d_E_SAR_diff_kobs14",
    "V_ema_LST_modis_kobs7",
    "A_d_LST_modis_kobs14",
    "V_rollmin_LST_modis_kobs30",
    "V_rollmean_s2_b11_kobs7",

    "A_d_F_NDMI_kobs7",

    "year_frac", "sin_year", "cos_year",
    "API_x_year", "SMAP_x_year",

    "slope", "elev",
    "K_slope_sin", "K_slope_cos", "K_aspect_cos",
    "J_clay_wfrac_b0", "J_sand_wfrac_b0",
]

FEATURE_COLS_DRY = [
    "SMAP_sm_pm_interp_ema02",
    "SMAP_sm_interp_grad7",
    "SMAP_sm_interp_diff1",
    "A_d_SMAP_sm_interp_kobs14",

    "V_ema_LST_modis_kobs7",
    "V_rollmin_LST_modis_kobs30",
    "A_d_LST_modis_kobs14",

    "slope", "elev",
    "K_slope_sin", "K_slope_cos", "K_aspect_cos",
    "J_clay_wfrac_b0", "J_sand_wfrac_b0",

    "G_API",
    "V_ema_G_API_kobs14",
    "C_lag_G_API_kobs1",

    "G_DSLR",

    "V_rollmean_s2_b11_kobs7",
    "year_frac", "sin_year", "cos_year",
]

FEATURE_COLS_WET = [
    "SMAP_sm_interp_diff1",
    "SMAP_sm_interp_rollstd7",
    "SMAP_sm_interp_rollrange7",
    "SMAP_sm_interp_pctchg",
    "A_d_SMAP_sm_interp_kobs7",
    "A_grad_SMAP_sm_interp_kobs7",
    "A_pct_SMAP_sm_interp",

    "SMAP_sm_interp_lag7",

    "G_API",
    "G_rain_sum_3d",
    "G_rain_sum_7d",

    "G_rain_sum_30d",

    "V_rollstd_G_API_kobs7",
    "V_rollcv_G_API_kobs7",
    "A_d_G_API_kobs7",

    "A_d_E_SAR_diff_kobs1",
    "A_d_E_SAR_diff_kobs7",
    "A_grad_E_SAR_diff_kobs7",
    "A_grad_E_SAR_ratio_kobs7",
    "V_rollstd_E_SAR_diff_kobs7",
    "V_rollstd_E_SAR_ratio_kobs7",

    "V_rollstd_F_NDMI_kobs7",
    "A_d_F_NDMI_kobs7",

    "year_frac", "sin_year", "cos_year",
    "slope", "elev",
]

def _check_cols(df, cols, name):
    missing = sorted(set(cols) - set(df.columns))
    if missing:
        raise ValueError(f"Missing columns in {name}: {missing}")

for _df_name, _df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    _check_cols(_df, KEEP_META_COLS + [TARGET_COL], _df_name)
    _check_cols(_df, FEATURE_COLS_BASE, f"{_df_name} (BASE)")
    _check_cols(_df, FEATURE_COLS_DRY, f"{_df_name} (DRY)")
    _check_cols(_df, FEATURE_COLS_WET, f"{_df_name} (WET)")

print("Columns locked")
print("  BASE features:", len(FEATURE_COLS_BASE))
print("  DRY  features:", len(FEATURE_COLS_DRY))
print("  WET  features:", len(FEATURE_COLS_WET))
print("  Target:", TARGET_COL)

Columns locked
  BASE features: 34
  DRY  features: 22
  WET  features: 28
  Target: soil_moisture_5cm


### Split Strategy
- **Training set:**  
  Two stations, early time period  
- **Validation set:**  
  Same stations as training, held-out **future dates** (temporal holdout)
- **Test set:**  
  One completely unseen station (station-level holdout)

### Motivation
- Validation evaluates **temporal generalization** on known stations
- Test evaluates **spatial generalization** to an unseen station
- This avoids spatial leakage while preserving sufficient training data

In [7]:
print("=== SPLIT SUMMARY ===")

def split_summary(name, d):
    print(f"\n{name.upper()}")
    print(f"  rows:     {len(d)}")
    print(f"  stations: {sorted(d['station_id'].unique().tolist())}")
    if "date" in d.columns:
        print(f"  date range: {d['date'].min()} -- {d['date'].max()}")

split_summary("train", train_df)
split_summary("val", val_df)
split_summary("test", test_df)

print("\n=== LEAKAGE CHECK ===")
print("train ∩ test:", sorted(set(train_df.station_id) & set(test_df.station_id)))
print("val   ∩ test:", sorted(set(val_df.station_id) & set(test_df.station_id)))

print("\n-- split locked --")


=== SPLIT SUMMARY ===

TRAIN
  rows:     29362
  stations: ['BeaverPass_WA_990', 'BurntMountain_WA', 'CayusePass_WA', 'Darrington', 'HartsPass_WA_515', 'MFNooksack_WA_1011', 'MartenRidge_WA_999', 'Paradise_WA', 'Quinault', 'RainyPass_WA_711', 'SCAN_ConradAgRc', 'SCAN_CookFarmFieldD', 'SCAN_JordanValleyCwma', 'SCAN_Lind_1', 'SCAN_OrchardRangeSite', 'SCAN_TableMountain', 'SCAN_Violett', 'SourdoughGulch_WA_985', 'Spokane', 'Touchet_WA_824', 'USCRN_Arco_17_SW', 'USCRN_Corvallis_10_SSW', 'USCRN_Darrington_21_NNE', 'USCRN_Dillon_18_WSW', 'USCRN_John_Day_35_WNW', 'USCRN_Murphy_10_W', 'USCRN_Quinault_4_NE', 'USCRN_Riley_10_WSW', 'USCRN_Spokane_17_SSW', 'USCRN_St_Mary_1_SSW']
  date range: 2017-01-01 -- 2020-12-31

VAL
  rows:     13637
  stations: ['BeaverPass_WA_990', 'BurntMountain_WA', 'CayusePass_WA', 'Darrington', 'HartsPass_WA_515', 'MartenRidge_WA_999', 'Paradise_WA', 'Quinault', 'RainyPass_WA_711', 'SCAN_ConradAgRc', 'SCAN_CookFarmFieldD', 'SCAN_JordanValleyCwma', 'SCAN_Lind_1', 'SCAN_

In [8]:
STATION = "station_id"
DATE = "date"

def add_dynamic_state(df, eps=0.01, smooth_window=3, zscore_per_station=False):
    d = df.copy()
    d[DATE] = pd.to_datetime(d[DATE])
    d = d.sort_values([STATION, DATE]).reset_index(drop=True)

    d["y_smooth"] = (
        d.groupby(STATION)[TARGET_COL]
         .rolling(smooth_window, min_periods=1)
         .mean()
         .reset_index(level=0, drop=True)
    )
    d["y_smooth_lag1"] = d.groupby(STATION)["y_smooth"].shift(1)
    d = d.dropna(subset=["y_smooth_lag1"]).reset_index(drop=True)

    d["dy"] = d["y_smooth"] - d["y_smooth_lag1"]

    if zscore_per_station:
        mu = d.groupby(STATION)["dy"].transform("mean")
        sd = d.groupby(STATION)["dy"].transform("std").replace(0, np.nan)
        d["dy_z"] = (d["dy"] - mu) / sd
        v = d["dy_z"].values
    else:
        v = d["dy"].values

    lbl = np.ones(len(d), dtype=int)
    lbl[v < -eps] = 0
    lbl[v >  eps] = 2
    d["dyn_lbl"] = lbl

    return d

In [9]:
eps = 1.0
smooth_window = 3
zscore_per_station = True

train_dyn = add_dynamic_state(
    train_df,
    eps=eps,
    smooth_window=smooth_window,
    zscore_per_station=zscore_per_station
)

val_dyn = add_dynamic_state(
    val_df,
    eps=eps,
    smooth_window=smooth_window,
    zscore_per_station=zscore_per_station
)

print("Train counts:", train_dyn["dyn_lbl"].value_counts().sort_index().to_dict())
print("Val counts:  ", val_dyn["dyn_lbl"].value_counts().sort_index().to_dict())

Train counts: {0: 2385, 1: 23907, 2: 3040}
Val counts:   {0: 1104, 1: 11152, 2: 1352}


In [10]:
EXCLUDE = {
    "station_id", "date",
    "soil_moisture_5cm",
    "dyn_lbl",
    "y_smooth",
    "y_smooth_lag1",
    "dy",
    "dy_z"
}

def get_feature_cols(df):
    cols = []
    for c in df.columns:
        if c in EXCLUDE:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)
    return cols

CLF_COLS = get_feature_cols(train_dyn)

print("Num classifier features:", len(CLF_COLS))
print("First 25:", CLF_COLS[:25])

Num classifier features: 496
First 25: ['longitude', 'latitude', 'precip_mm', 's1_vv', 's1_vh', 's2_b4', 's2_b8', 's2_b11', 's2_b12', 'LST_modis', 'elev', 'slope', 'aspect', 'DOY', 'SMAP_sm_am_interp', 'SMAP_sm_pm_interp', 'J_aspect_deg', 'J_bio_bio01', 'J_bio_bio02', 'J_bio_bio03', 'J_bio_bio04', 'J_bio_bio05', 'J_bio_bio06', 'J_bio_bio07', 'J_bio_bio08']


In [11]:
def clean_numeric_df(X):
    return X.replace([np.inf, -np.inf], np.nan)

X_tr = train_dyn[CLF_COLS].copy()
y_tr = train_dyn["dyn_lbl"].values

X_va = val_dyn[CLF_COLS].copy()
y_va = val_dyn["dyn_lbl"].values

X_tr = clean_numeric_df(X_tr)
X_va = clean_numeric_df(X_va)

clip_q = 0.001
if clip_q:
    lo = X_tr.quantile(clip_q, axis=0, numeric_only=True)
    hi = X_tr.quantile(1 - clip_q, axis=0, numeric_only=True)
    X_tr = X_tr.clip(lower=lo, upper=hi, axis=1)
    X_va = X_va.clip(lower=lo, upper=hi, axis=1)

imputer = SimpleImputer(strategy="median")
X_tr_i = imputer.fit_transform(X_tr)
X_va_i = imputer.transform(X_va)

print("train NaNs:", int(np.isnan(X_tr.values).sum()), "infs:", int(np.isinf(X_tr.values).sum()))
print("val   NaNs:", int(np.isnan(X_va.values).sum()), "infs:", int(np.isinf(X_va.values).sum()))
print("after impute train NaNs:", int(np.isnan(X_tr_i).sum()))
print("after impute val   NaNs:", int(np.isnan(X_va_i).sum()))

train NaNs: 9694220 infs: 0
val   NaNs: 4638943 infs: 0
after impute train NaNs: 0
after impute val   NaNs: 0


In [12]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
from xgboost import XGBClassifier

classes = np.array([0, 1, 2])
cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
cw_map = dict(zip(classes, cw))
sw = np.array([cw_map[int(c)] for c in y_tr], dtype=float)

xgb_clf = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=SEED,
    n_jobs=-1,
    max_depth=4,
    min_child_weight=5,
    subsample=0.9,
    colsample_bytree=0.9,
    n_estimators=8000,
    learning_rate=0.01,
    reg_lambda=2.0,
    reg_alpha=0.0,
)

xgb_clf.fit(
    X_tr_i, y_tr,
    sample_weight=sw,
    eval_set=[(X_va_i, y_va)],
    verbose=200
)

proba_va = xgb_clf.predict_proba(X_va_i)

p_dry = proba_va[:, 0]
p_stable = proba_va[:, 1]
p_wet = proba_va[:, 2]

stable_thresh = 0.60

pred_va = np.full(len(proba_va), 1, dtype=int)
mask_not_stable = p_stable < stable_thresh

pred_va[mask_not_stable] = np.argmax(
    np.column_stack([p_dry[mask_not_stable], p_stable[mask_not_stable], p_wet[mask_not_stable]]),
    axis=1
)

[0]	validation_0-mlogloss:1.09692
[200]	validation_0-mlogloss:0.98306
[400]	validation_0-mlogloss:0.94188
[600]	validation_0-mlogloss:0.92021
[800]	validation_0-mlogloss:0.90471


KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_va, pred_va))
print("Balanced accuracy:", balanced_accuracy_score(y_va, pred_va))
print(classification_report(y_va, pred_va, target_names=["drying","stable","wetting"], digits=4))

cm = confusion_matrix(y_va, pred_va, labels=[0,1,2])
cm_df = pd.DataFrame(cm,
                     index=["true_drying","true_stable","true_wetting"],
                     columns=["pred_drying","pred_stable","pred_wetting"])
print(cm_df)

Accuracy: 0.621766607877719
Balanced accuracy: 0.46096041893847106
              precision    recall  f1-score   support

      drying     0.1838    0.3732    0.2463      1104
      stable     0.8739    0.6820    0.7661     11152
     wetting     0.1664    0.3277    0.2207      1352

    accuracy                         0.6218     13608
   macro avg     0.4080    0.4610    0.4110     13608
weighted avg     0.7476    0.6218    0.6698     13608

              pred_drying  pred_stable  pred_wetting
true_drying           412          496           196
true_stable          1523         7606          2023
true_wetting          307          602           443


In [ ]:
import joblib

MODEL_PATH = "v20/v20.4/regime_classifier_xgb_d9.json"
IMPUTER_PATH = "v20/v20.4/stable_classifier_imputer_d9.pkl"
xgb_clf.save_model(MODEL_PATH)
joblib.dump(CLF_COLS, "v20/v20.4/stable_classifier_cols_d9.pkl")
joblib.dump(imputer, IMPUTER_PATH)


['v20/v20.4/stable_classifier_imputer_d9.pkl']

In [ ]:
imp = xgb_clf.feature_importances_
imp_df = pd.DataFrame({"feature": CLF_COLS, "importance": imp}).sort_values("importance", ascending=False)
print(imp_df.head(10).to_string(index=False))

                       feature  importance
      C_smm_G_API_alpha0.85_n5    0.011253
              J_clay_wfrac_b60    0.011026
       V_rollmax_F_NDVI_kobs30    0.010917
            V_ema_F_NDMI_kobs7    0.010844
                         F_MSI    0.010807
                  J_aspect_deg    0.010722
                   SMAP_x_year    0.009218
V_rollcv_SMAP_sm_interp_kobs30    0.008250
       V_rollmin_F_NDMI_kobs30    0.007834
       V_rollmax_F_NDVI_kobs14    0.007830
